# Setting up OD Trade-Flows

In this notebook we generate the data-input file od.csv 

# Import libraries

In [44]:
import pandas as pd
import numpy as np
import geopandas as gpd
import duckdb

# Inferring OD Trade-flows

In [45]:
### This needs to be rewritten before being uploaded to Github.
port_trade_network_df = pd.read_csv('./raw_data/od/port_trade_network.csv')

port_trade_network_reduced_df = port_trade_network_df[['from_id', 'to_id', 'flow', 'iso3_O', 'iso3_D','q_share_port', 'v_share_port','q_sea_flow', 'v_sea_flow', 'Industries']].copy()


In [46]:
port_trade_network_reduced_df['port'] = port_trade_network_reduced_df['from_id'].str.split('_', expand=True)[0]


In [47]:
con = duckdb.connect()
con.register("network", port_trade_network_reduced_df)


In [48]:
con.execute("""
CREATE OR REPLACE TABLE exports AS
SELECT
    port AS origin_portid,
    iso3_O,
    iso3_D,
    Industries,
    v_sea_flow,
    q_sea_flow
FROM network
WHERE flow = 'port_export'
""")

con.execute("""
CREATE OR REPLACE TABLE imports AS
SELECT
    port AS destination_portid,
    iso3_O,
    iso3_D,
    Industries,
    v_share_port,
    q_share_port
FROM network
WHERE flow = 'port_import'
""")

con.execute("""
CREATE OR REPLACE TABLE export_import_pairs AS
SELECT
    e.origin_portid,
    i.destination_portid,
    e.iso3_O,
    e.iso3_D,
    e.Industries,
    e.v_sea_flow,
    e.q_sea_flow,
    i.v_share_port,
    i.q_share_port
FROM exports e
JOIN imports i
  ON e.iso3_O = i.iso3_O
 AND e.iso3_D = i.iso3_D
 AND e.Industries = i.Industries
""")


con.execute("""
CREATE OR REPLACE TABLE export_import_pairs_final AS
SELECT
    origin_portid,
    destination_portid,
    iso3_O,
    iso3_D,
    Industries,
    v_sea_flow * v_share_port AS value_exports,
    q_sea_flow * q_share_port AS qty_exports
FROM export_import_pairs
""")

#### This Output folder below needs to be re-inspected
con.execute("""
COPY export_import_pairs_final
TO 'raw_data/od/intermediate_data/Worldwide_port_export_vol.parquet'
(FORMAT PARQUET)
""")


## Re-arrange trade data to produce od.csv

In [49]:
trade_data = pd.read_parquet('./raw_data/od/intermediate_data/Worldwide_port_export_vol.parquet')

In [50]:
trade_data = trade_data.groupby(['origin_portid','destination_portid','iso3_O','iso3_D','Industries']).sum().reset_index()

In [51]:
trade_data_wide = (
    trade_data
    .drop(columns="value_exports")
    .pivot(
        index=["origin_portid", "destination_portid", "iso3_O", "iso3_D"],
        columns="Industries",
        values="qty_exports"
    )
    .reset_index()
)

trade_data_wide.columns.name = None

In [52]:
trade_data_wide = trade_data_wide.fillna(0)

In [53]:
trade_data_wide = trade_data_wide.rename(columns = {'origin_portid':'origin_id','destination_portid':'destination_id'})

In [54]:
industry_cols = [i for i in range(1, 12)]

# Rename 1, 2, ..., 11 to Industry_1, Industry_2, ..., Industry_11
trade_data_wide = trade_data_wide.rename(
    columns={col: f"Industry_{col}" for col in industry_cols}
)

# Sum all industry columns for each row
trade_data_wide["tons"] = trade_data_wide[
    [f"Industry_{i}" for i in range(1, 12)]
].sum(axis=1)

In [55]:
trade_data_wide.to_csv("./processed_data/od/od.csv", index=False)